# Célula 1 — Markdown
"""
## 5. Python Client

Para facilitar a reprodutibilidade dos experimentos e uso prático dos modelos,
desenvolvemos um cliente Python unificado com interface CLI.
"""

In [ ]:
# Célula 2 — Classe unificada
class ASRClient:
    """
    Cliente unificado para SeamlessM4T e NeMo Canary.
    Interface comum independente do backend.
    """
    MODELOS_SUPORTADOS = ["seamless", "canary"]

    def __init__(self, modelo: str = "seamless"):
        assert modelo in self.MODELOS_SUPORTADOS
        self.modelo = modelo
        self._carregar_modelo()

    def _carregar_modelo(self):
        if self.modelo == "seamless":
            self.processor = AutoProcessor.from_pretrained("facebook/seamless-m4t-v2-large")
            self.net = SeamlessM4Tv2Model.from_pretrained("facebook/seamless-m4t-v2-large")
        else:
            self.net = nemo_asr.models.EncDecMultiTaskModel.from_pretrained("nvidia/canary-1b")

    def transcrever(self, audio_path: str,
                    src_lang: str = "por", tgt_lang: str = "eng") -> dict:
        t0 = time.perf_counter()

        if self.modelo == "seamless":
            audio, sr = torchaudio.load(audio_path)
            inputs = self.processor(audios=audio, sampling_rate=sr,
                                    src_lang=src_lang, tgt_lang=tgt_lang,
                                    return_tensors="pt")
            with torch.no_grad():
                out = self.net.generate(**inputs, tgt_lang=tgt_lang)
            texto = self.processor.decode(out[0].tolist(), skip_special_tokens=True)
        else:
            resultado = self.net.transcribe(audio=[audio_path], batch_size=1,
                                            task="s2t_translation",
                                            source_lang=src_lang[:2],
                                            target_lang=tgt_lang[:2])
            texto = resultado[0].text

        return {
            "modelo":     self.modelo,
            "texto":      texto,
            "latencia_s": round(time.perf_counter() - t0, 3),
            "src_lang":   src_lang,
            "tgt_lang":   tgt_lang,
        }

# Teste rápido
cliente = ASRClient(modelo="seamless")
resultado = cliente.transcrever("data/sample.wav")
print(resultado)